In [ ]:
!pip install langchain_core
!pip install langchain_google_genai
!pip install grandalf

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

gemini_api = userdata.get('gemini_api')
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=gemini_api
)

*Chains: It is a pipeline that can *intialize* prompt, model.. together*

**Sequential chain**

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

sequential_prompt1 = PromptTemplate(
    template = 'write a summary about {topic}',
    input_variables = ['topic']
)

sequential_prompt2 = PromptTemplate(
    template = 'Give 5 important points from summary -> {summary}',
    input_variables = ['summary']
)

str_parser = StrOutputParser()

seq_chain = sequential_prompt1 | model | str_parser | sequential_prompt2 | model | str_parser

seq_Chain_result = seq_chain.invoke({'topic':'Artificial Intelligence'})
print(seq_Chain_result)

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 3.302696781s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

**Parallel chain**

In [ ]:
model2 = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=gemini_api
)

In [ ]:
from langchain_core.prompts import PromptTemplate

summary_template = PromptTemplate(
    template = 'write a summary for the given text\n {text}',
    input_variables = ['text']
)

quiz_template = PromptTemplate(
    template = 'Create a quiz with questions and answers on the given text\n {text}',
    input_variables = ['text']
)

merge_template = PromptTemplate(
    template = 'merge the summary text -> {summary} and quiz -> {quiz}',
    input_variables = ['summary','quiz']
)

In [ ]:
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel(
    summary = summary_template | model | str_parser,
    quiz = quiz_template | model2 | str_parser
)

merge_chain = parallel_chain | merge_template | model | str_parser

In [ ]:
text = """Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64"""

In [ ]:
parallel_chain_output = parallel_chain.invoke({'text':text})
print(parallel_chain_output)

In [ ]:

parallel_chain.get_graph().print_ascii()

**Conditional chain**

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

class Sentiment(BaseModel):
  sentiment: Literal['positive','negative'] = Field(description='sentiment of the text')

sentiment_parser = PydanticOutputParser(pydantic_object=Sentiment)

In [ ]:

classifier_template = PromptTemplate(
    template = 'Classify the sentiment of the feedback as positive or negative\n {text} {output_parser}',
    input_variables = ['text'],
    partial_variables={'output_parser':sentiment_parser.get_format_instructions()}
)

positive_template = PromptTemplate(
    template = 'write a summary about the following positive feedback\n {text}',
    input_variables = ['text']
)

negative_template = PromptTemplate(
    template = 'write a summary about the following negative feedback\n {text}',
    input_variables = ['text']
)


In [ ]:
gemini_api2 = userdata.get('gemini_api2')
model3 = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=gemini_api2
)

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnablePassthrough



classifier_chain = classifier_template | model3 | sentiment_parser

conditional_chain = (
    RunnablePassthrough.assign(
        sentiment_result=classifier_chain
    )
    | RunnableBranch(

        (lambda x: x['sentiment_result'].sentiment == 'positive',
         # For the positive branch, we need to pass only the original 'text' to the positive_template.
         # We use RunnableLambda to select x['text'] and format it as {'text': x['text']}.
         RunnableLambda(lambda x_branch: {'text': x_branch['text']}) | positive_template | model3 | str_parser),
        (lambda x: x['sentiment_result'].sentiment == 'negative',
         # Similarly for the negative branch.
         RunnableLambda(lambda x_branch: {'text': x_branch['text']}) | negative_template | model3 | str_parser),
        # Fallback for when sentiment is not clear. This lambda just returns a string.
        RunnableLambda(lambda x_branch: "Sentiment is not clear")
    )
)


In [ ]:
conditional_chain.invoke({'text':'This is a great mobile'})

'**Summary:** \n\nThe customer provided highly positive feedback, expressing overall satisfaction and describing the mobile device as "great."'

In [ ]:
conditional_chain.invoke({'text':'This is a terrible TV'})

'The customer expressed strong dissatisfaction, stating that the television is "terrible." However, the feedback is highly general and does not provide any specific details or reasons for their disappointment.'

In [ ]:
conditional_chain.get_graph().print_ascii()

             +---------------------------------+         
             | Parallel<sentiment_result>Input |         
             +---------------------------------+         
                     ***              ***                
                  ***                    ***             
                **                          ***          
    +----------------+                         **        
    | PromptTemplate |                          *        
    +----------------+                          *        
             *                                  *        
             *                                  *        
             *                                  *        
+------------------------+                      *        
| ChatGoogleGenerativeAI |                      *        
+------------------------+                      *        
             *                                  *        
             *                                  *        
             *